# logit-thermo A2-6F：しらふ同士の床と、家族の差の天井（CPU）

- **必要な secret：なし**（`HF_TOKEN` があればトークナイザーの取り込みが少し速くなるだけ）。**ランタイム：CPU（標準）**。見込みは 20〜30 分。
- **A2-6 のデータ集め 3 本と、A2-6 の分析が終わってから**回してください（A2-6 の要約を読みます）。
- ゲート：`docs/A2_6F_GATE.md`。床を 3 種類（種だけ違う 4 組／1 回の中の 2 本のサンプルを分けた 25 組／記事を 20 本ずつに分けた 25 組）出し、`A2_6/a2_6f_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：A2-2・探索 C・A2-5 のコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_6f_floor.py`（`3b302260eb12018d…`）は埋め込み（データを開く前に固定）。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo A2-6F 分析（しらふ同士の床と天井、CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-6F 分析（しらふ同士の床と天井、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-6F floor', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：分析のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRCA22': 'fe1f96db1f7e84031e85a70bcc8dcbb95c9fc06b3367657ff4074a9f9d1343f7', 'SRCX': '012c6dc1fb2ac8a3d120fa2c35a14480c7dc82815e6775cb3829a8bf16284759', 'SRC5A': 'f4ef35b9f594a205ac1145c1a326e9de8a25280171fe2b93083cc1676e58d3ae'})
SRC22, SRCA22, SRCX, SRC5A = _s['SRC22'], _s['SRCA22'], _s['SRCX'], _s['SRC5A']
SRC6F = '"""A2-6F（ゲート docs/A2_6F_GATE.md）：しらふ同士の床と、家族の差の天井。データを開く前に固定する。CPU。\nscripts/a2_2_collect.py（split_doc）・scripts/a2_2_analyze.py（per_doc_feedback・feedback_stats）・\nscripts/a2_explore_copy.py（copy_flags・decompose）・scripts/a2_5_analyze.py（doc_parts・G_at）を実行したのと同じ名前空間で実行する。\n\n床は 3 種類（どれも GPU を使わず、手元のデータから出す）：\n- 組の床：同じモデル・同じ記事・同じコードで、生成の種だけ違う 2 回の差（A2-2 の種 20261027 と、A2-5/A2-6 の種 20261102）。4 組。\n- 種の半分の床：1 回の中の 2 本のサンプル（1 本目と 2 本目）を分けて、それぞれで測った差。A2-2・A2-5・A2-6 の全部の回。\n- 記事の半分の床：40 本の記事を、使った順で偶数番目と奇数番目の 20 本ずつに分けて測った差（G07 は 30 本以上ないと読まないので出さない）。\n半分どうしの差は、1 回まるごとどうしの差よりおよそ √2 倍大きくなる（半分は雑音の分散が 2 倍）ので、√2 で割って比べる。\n    analyze6f(logit-thermo のフォルダ, docs) -> dict\n"""\nimport json, math\nimport numpy as np\n\nF6_PAIRS = [  # (名前, 古い種のフォルダ, 新しい種のフォルダ, トークナイザー)\n    (\'pythia-1b\', \'A2_2\', \'A2_5\', \'EleutherAI/pythia-1b\'),\n    (\'qwen2.5-0.5b\', \'A2_2\', \'A2_6\', \'Qwen/Qwen2.5-0.5B\'),\n    (\'qwen2.5-1.5b\', \'A2_2\', \'A2_6\', \'Qwen/Qwen2.5-1.5B\'),\n    (\'smollm2-1.7b\', \'A2_2\', \'A2_6\', \'HuggingFaceTB/SmolLM2-1.7B\'),\n]\nF6_RUNS = {  # 半分の床に使う全部の回：フォルダ → [(名前, トークナイザー)]\n    \'A2_2\': [(\'pythia-1b\', \'EleutherAI/pythia-1b\'), (\'qwen2.5-0.5b\', \'Qwen/Qwen2.5-0.5B\'), (\'qwen2.5-0.5b-inst\', \'Qwen/Qwen2.5-0.5B-Instruct\'),\n             (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\'), (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'), (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\'),\n             (\'smollm2-1.7b-inst\', \'HuggingFaceTB/SmolLM2-1.7B-Instruct\')],\n    \'A2_5\': [(n, \'EleutherAI/pythia-1b\') for n in [\'pythia-70m\', \'pythia-160m\', \'pythia-410m\', \'pythia-1b\', \'pythia-1.4b\', \'pythia-2.8b\',\n                                                   \'pythia-1b@step8000\', \'pythia-1b@step24000\', \'pythia-1b@step64000\']],\n    \'A2_6\': [(\'qwen2.5-0.5b\', \'Qwen/Qwen2.5-0.5B\'), (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\'), (\'qwen2.5-3b\', \'Qwen/Qwen2.5-3B\'),\n             (\'smollm2-135m\', \'HuggingFaceTB/SmolLM2-135M\'), (\'smollm2-360m\', \'HuggingFaceTB/SmolLM2-360M\'),\n             (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\'), (\'qwen3-0.6b\', \'Qwen/Qwen3-0.6B-Base\'), (\'qwen3-1.7b\', \'Qwen/Qwen3-1.7B-Base\'),\n             (\'qwen3-4b\', \'Qwen/Qwen3-4B-Base\')],\n}\nF6_KEYS = [\'T0_logratio\', \'F07\', \'s\', \'G07\']\nF6_STEPS = [\'pythia-1b@step8000\', \'pythia-1b@step24000\', \'pythia-1b@step64000\', \'pythia-1b\']\nF6_EQ = 1.05            # A2-6 の Q9「動かない」の線（家族の中の T_0 の最大 ÷ 最小）\nF6_CEIL_X = 3.0         # 天井が床の何倍あれば「家族の差は床よりずっと大きい」と言うか\n\n\ndef G_one_temp(Z, prompts, humans, k):\n    """A2-5 の doc_parts と G_at を、温度 k だけについて同じ手順で計算する（速くするため）。"""\n    toks, gen, tf = Z[\'toks\'], Z[\'gen\'][0], Z[\'tf\'][0]\n    nT, S, nD, L = toks.shape\n    a, b = A5_BAND\n    g, f = [], []\n    for j in range(nD):\n        hum = copy_flags(prompts[j], humans[j])\n        gfl = np.stack([copy_flags(prompts[j], toks[k, s, j]) for s in range(S)])\n        r = decompose(gen[k, :, j, a:b], gfl[:, a:b], tf[j, k, a:b], hum[a:b])\n        if r is not None:\n            g.append(r[\'glass\']); f.append(r[\'f\'])\n    n = len(f)\n    if n == 0:\n        return float(\'nan\'), float(\'nan\'), 0\n    mg, mf = float(np.mean(g)), float(np.mean(f))\n    return (mg / mf if (n >= A5_MIN_DOCS and mf <= A5_FMAX) else float(\'nan\')), mf, n\n\n\ndef sub(Z, samples=None, docs=None):\n    """サンプルや記事を選んだ Z（tf：(4, nD, nT, L)、gen：(4, nT, S, nD, L)、toks：(nT, S, nD, L)）。"""\n    Z2 = dict(Z)\n    if samples is not None:\n        Z2[\'gen\'] = Z2[\'gen\'][:, :, samples]; Z2[\'toks\'] = Z2[\'toks\'][:, samples]\n    if docs is not None:\n        Z2[\'tf\'] = Z2[\'tf\'][:, docs]; Z2[\'gen\'] = Z2[\'gen\'][:, :, :, docs]; Z2[\'toks\'] = Z2[\'toks\'][:, :, docs]\n    return Z2\n\n\ndef measures(Z, prompts, humans, with_G=True):\n    temps = [float(t) for t in Z[\'temps\']]\n    k07 = temps.index(0.7)\n    fd = per_doc_feedback(Z[\'tf\'][0], Z[\'gen\'][0])\n    fs = feedback_stats(fd, temps, np.random.default_rng(0))\n    F = np.median(fd, 0)\n    G07, f07, n07 = G_one_temp(Z, prompts, humans, k07) if with_G else (float(\'nan\'), float(\'nan\'), 0)\n    return dict(T0=fs[\'T0\'], F07=float(F[k07]), s=float((F[temps.index(0.9)] - F[temps.index(0.8)]) / 0.1),\n                G07=G07, f07_band=f07, n07=n07)\n\n\ndef diffs(a, b):\n    lr = abs(math.log(a[\'T0\'] / b[\'T0\'])) if (np.isfinite(a[\'T0\']) and np.isfinite(b[\'T0\'])) else float(\'nan\')\n    return dict(T0_logratio=lr, F07=abs(a[\'F07\'] - b[\'F07\']), s=abs(a[\'s\'] - b[\'s\']), G07=abs(a[\'G07\'] - b[\'G07\']))\n\n\ndef p95(v):\n    v = np.asarray(v, float); v = v[np.isfinite(v)]\n    return float(np.percentile(v, 95)) if len(v) else float(\'nan\')\n\n\ndef floor_from_vals(vals, a25, a26, halves_seed=None, halves_doc=None):\n    """vals：{名前: (古い種の measures, 新しい種の measures)}。a25・a26：A2-5・A2-6 の要約（dict）。\n    halves_seed・halves_doc：{回の名前: (半分 1 の measures, 半分 2 の measures)}。"""\n    res = {\'pairs\': {}}\n    d = {k: [] for k in F6_KEYS}\n    for n, (a, b) in vals.items():\n        row = dict(old=a, new=b, **diffs(a, b))\n        res[\'pairs\'][n] = row\n        for k in d:\n            d[k].append(row[k])\n    res[\'pair_floor_max\'] = {k: float(np.nanmax(v)) for k, v in d.items()}\n    res[\'pair_floor_median\'] = {k: float(np.nanmedian(v)) for k, v in d.items()}\n    judge = dict(res[\'pair_floor_max\'])\n    for tag, hv in ((\'seed_half\', halves_seed or {}), (\'doc_half\', halves_doc or {})):\n        rows = {n: diffs(a, b) for n, (a, b) in hv.items()}\n        res[tag] = rows\n        res[tag + \'_p95\'] = {k: p95([r[k] for r in rows.values()]) for k in F6_KEYS}\n        res[tag + \'_max\'] = {k: (float(np.nanmax([r[k] for r in rows.values()])) if rows and np.isfinite([r[k] for r in rows.values()]).any() else float(\'nan\')) for k in F6_KEYS}\n        res[tag + \'_n_finite\'] = {k: int(np.isfinite([r[k] for r in rows.values()]).sum()) for k in F6_KEYS}\n        for k in F6_KEYS:\n            v = res[tag + \'_p95\'][k] / math.sqrt(2)\n            if np.isfinite(v):\n                judge[k] = max(judge[k], v) if np.isfinite(judge[k]) else v\n    res[\'floor_max\'] = judge          # 判定に使う床：組の床のいちばん大きい値と、半分の床の 95% 点 ÷ √2 のうち大きいほう\n    # F1：T_0 の床は、A2-6 の「動かない」の線（5%）の内側か\n    fr = math.exp(res[\'floor_max\'][\'T0_logratio\'])\n    res[\'F1\'] = \'床は 5% の線の内側\' if fr < F6_EQ else \'床が 5% の線に届く\'\n    res[\'F1_detail\'] = dict(floor_ratio=fr, line=F6_EQ)\n    # F2：家族の差の天井（A2-6 の家族ごとの T_0 の幾何平均の、いちばん離れた 2 家族）は床の何倍か\n    gm = {g: v[\'geo_mean\'] for g, v in (a26 or {}).get(\'fam_T0\', {}).items()}\n    if len(gm) >= 2 and res[\'floor_max\'][\'T0_logratio\'] > 0:\n        ceil = math.log(max(gm.values()) / min(gm.values()))\n        x = ceil / res[\'floor_max\'][\'T0_logratio\']\n        res[\'F2\'] = \'天井は床の 3 倍以上\' if x >= F6_CEIL_X else \'天井は床の 3 倍より小さい\'\n        res[\'F2_detail\'] = dict(ceiling_logratio=ceil, times_floor=x, fam_geo_mean=gm)\n    else:\n        res[\'F2\'] = \'読めない\'; res[\'F2_detail\'] = dict(fam_geo_mean=gm)\n    # F3：A2-5 の学びの軸で、隣どうしの G07 の差は床（G07 の最大）を超えるか\n    G = [(a25 or {}).get(\'models\', {}).get(n, {}).get(\'G07\', float(\'nan\')) for n in F6_STEPS]\n    steps = []\n    for i in range(len(G) - 1):\n        diff = G[i + 1] - G[i]\n        w = \'床の上\' if (np.isfinite(diff) and abs(diff) > res[\'floor_max\'][\'G07\']) else (\'床の中\' if np.isfinite(diff) else \'読めない\')\n        steps.append(dict(frm=F6_STEPS[i], to=F6_STEPS[i + 1], diff=float(diff), word=w))\n    res[\'F3_steps\'] = steps\n    res[\'F3\'] = sum(s[\'word\'] == \'床の上\' for s in steps)\n    first, last = G[0], G[-1]\n    res[\'F3_whole\'] = dict(diff=float(last - first), word=\'床の上\' if abs(last - first) > res[\'floor_max\'][\'G07\'] else \'床の中\')\n    return res\n\n\ndef analyze6f(base, docs):\n    from transformers import AutoTokenizer\n    vals = {}\n    for n, da, db, hf in F6_PAIRS:\n        tok = AutoTokenizer.from_pretrained(hf)\n        pairs = [split_doc(tok, d[\'text\']) for d in docs]\n        Zs = [(lambda z: {k: z[k] for k in z.files})(np.load(f\'{base}/{dd}/{n}.npz\')) for dd in (da, db)]\n        uses = [[int(u) for u in Z[\'use\']] for Z in Zs]\n        assert uses[0] == uses[1], (\'使った記事が 2 回で違います\', n)\n        prompts = [pairs[i][0] for i in uses[0]]; humans = [pairs[i][1] for i in uses[0]]\n        vals[n] = tuple(measures(Z, prompts, humans) for Z in Zs)\n    a25 = json.load(open(f\'{base}/A2_5/a2_5_summary.json\'))\n    try:\n        a26 = json.load(open(f\'{base}/A2_6/a2_6_summary.json\'))\n    except FileNotFoundError:\n        a26 = None\n    hs, hd = {}, {}\n    for folder, runs in F6_RUNS.items():\n        for n, hf in runs:\n            tok = AutoTokenizer.from_pretrained(hf)\n            pairs = [split_doc(tok, d[\'text\']) for d in docs]\n            Z = (lambda z: {k: z[k] for k in z.files})(np.load(f\'{base}/{folder}/{n}.npz\'))\n            use = [int(u) for u in Z[\'use\']]\n            prompts = [pairs[i][0] for i in use]; humans = [pairs[i][1] for i in use]\n            assert Z[\'gen\'].shape[2] == 2, (\'サンプルが 2 本ではありません\', folder, n)\n            hs[f\'{folder}/{n}\'] = tuple(measures(sub(Z, samples=[s_]), prompts, humans) for s_ in (0, 1))\n            ev, od = list(range(0, len(use), 2)), list(range(1, len(use), 2))\n            hd[f\'{folder}/{n}\'] = tuple(measures(sub(Z, docs=ix), [prompts[i] for i in ix], [humans[i] for i in ix], with_G=False)\n                                        for ix in (ev, od))\n    return floor_from_vals(vals, a25, a26, hs, hd)\n\n\ndef self_test_a26f():\n    mk = lambda T0, F07, s, G: dict(T0=T0, F07=F07, s=s, G07=G, f07_band=-1.0, n07=40)\n    vals = {\'pythia-1b\': (mk(0.852, -0.88, 3.9, 0.29), mk(0.841, -0.90, 4.1, 0.33)),\n            \'qwen2.5-0.5b\': (mk(0.858, -2.0, 6.4, 0.40), mk(0.866, -2.1, 6.0, 0.41))}\n    a25 = {\'models\': {\'pythia-1b@step8000\': {\'G07\': 0.20}, \'pythia-1b@step24000\': {\'G07\': 0.27},\n                      \'pythia-1b@step64000\': {\'G07\': 0.30}, \'pythia-1b\': {\'G07\': 0.33}}}\n    a26 = {\'fam_T0\': {\'A\': {\'geo_mean\': 0.80}, \'B\': {\'geo_mean\': 0.90}}}\n    r = floor_from_vals(vals, a25, a26)\n    assert abs(r[\'floor_max\'][\'G07\'] - 0.04) < 1e-9, r[\'floor_max\']\n    # 半分の床が組の床より大きければ、判定にはそちら（95% 点 ÷ √2）を使う\n    hs = {f\'r{i}\': (mk(0.85, -1.0, 4.0, 0.30), mk(0.85, -1.0, 4.0, 0.30 + 0.01 * i)) for i in range(20)}\n    r2 = floor_from_vals(vals, a25, a26, halves_seed=hs)\n    assert abs(r2[\'floor_max\'][\'G07\'] - max(0.04, np.percentile([0.01 * i for i in range(20)], 95) / math.sqrt(2))) < 1e-9, r2[\'floor_max\']\n    # 交点が出ない半分は外して数える\n    hd = {\'x\': (mk(float(\'nan\'), -1.0, 4.0, float(\'nan\')), mk(0.85, -1.0, 4.0, float(\'nan\')))}\n    r3 = floor_from_vals(vals, a25, a26, halves_doc=hd)\n    assert r3[\'doc_half_n_finite\'][\'T0_logratio\'] == 0 and r3[\'floor_max\'][\'T0_logratio\'] == r3[\'pair_floor_max\'][\'T0_logratio\']\n    assert r[\'F1\'] == \'床は 5% の線の内側\', r[\'F1_detail\']\n    assert r[\'F2\'] == \'天井は床の 3 倍以上\', r[\'F2_detail\']\n    assert r[\'F3\'] == 1 and r[\'F3_steps\'][0][\'word\'] == \'床の上\' and r[\'F3_steps\'][1][\'word\'] == \'床の中\', r[\'F3_steps\']\n    assert r[\'F3_whole\'][\'word\'] == \'床の上\'\n    assert floor_from_vals(vals, a25, None)[\'F2\'] == \'読めない\'\n    return \'OK\'\n'
assert hashlib.sha256(SRC6F.encode()).hexdigest() == '3b302260eb12018d9a553723b61915c5c341e744c10a80f830bd4d4c35e1affd'
exec(SRC22)
exec(SRCA22)
exec(SRCX)
exec(SRC5A)
exec(SRC6F)
print('自己テスト：', self_test_a26f())


In [ ]:
# セル3：分析して保存（結果は表示しない）
import json
try:
    # A2-2 の分析のコード（SRCA22）が SEED を上書きするので、記事を選ぶあいだだけ A2-2 の種（20261027）にする（A2-5 の分析と同じ）
    _seed_keep = SEED
    SEED = 20261027
    docs, meta = load_docs()
    SEED = _seed_keep
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    r = analyze6f('/content/drive/MyDrive/logit-thermo', docs)
    r['a2_6f_floor_sha256'] = '3b302260eb12018d9a553723b61915c5c341e744c10a80f830bd4d4c35e1affd'
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo' + '/A2_6/a2_6f_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-6F の分析で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：床と天井を A2_6/a2_6f_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
